### Bibliotecas

In [30]:
import pandas as pd
import numpy as np
from pathlib import Path

# Imputação baseada em ML
from sklearn.experimental import enable_iterative_imputer  # noqa: F401 — necessário para habilitar o IterativeImputer
from sklearn.impute import IterativeImputer
from sklearn.ensemble import ExtraTreesRegressor

# Aumento de dados
from imblearn.over_sampling import SMOTE

import warnings
warnings.filterwarnings("ignore")

### Configuração Global

In [ ]:
TARGET_ROWS    = 3_000_000  # tamanho-alvo da base final
RANDOM_STATE   = 42         # semente para reproducibilidade
IMPUTER_SAMPLE = 200_000    # linhas usadas para *treinar* o imputer (fit rápido)
NOISE_SCALE    = 0.01       # magnitude do ruído gaussiano (1% do std de cada coluna)

# Colunas numéricas que serão imputadas e usadas no augmentation
# Definido aqui para estar disponível em qualquer célula do notebook
IMPUTE_COLS = ["BER", "OSNR", "InputPower", "OutputPower",
               "LP_length_km", "Laser_current_mA", "LP_power_dBm"]

### Carregando Dados

In [33]:
DATA_DIR   = Path("../Datasets")
OUTPUT_DIR = Path("../Datasets/output")
OUTPUT_DIR.mkdir(exist_ok=True)

# Hard Failure
# Telemetria de amplificadores e transponders com rótulo binário (0=Normal, 1=Falha)
df_hard = pd.read_csv(DATA_DIR / "HardFailure_dataset.csv")
df_hard["source"] = "hard_failure"

# Soft Failure
# Mesma estrutura do Hard Failure; captura degradações graduais de sinal
df_soft = pd.read_csv(DATA_DIR / "SoftFailure_dataset.csv")
df_soft["source"] = "soft_failure"

# Lightpath QoT
# Dataset de Qualidade de Transmissão com 4 tipos de falha (0=Normal, 1=ECL, 2=EDFA, 3=NLI)
# Formato: espaço-delimitado, 2 linhas de cabeçalho a pular
df_lp = pd.read_csv(
    DATA_DIR / "Lightpath_756_label_4_QoT_dataset_train_900.txt",
    sep=r"\s+",
    skiprows=2,
    header=None,
    names=["Timestamp", "LP_length_km", "Laser_current_mA", "LP_power_dBm", "OSNR", "BER_dB", "Failure_type"]
)
df_lp["source"] = "lightpath"

print(f"HardFailure : {df_hard.shape[0]:>10,} linhas | {df_hard.shape[1]} colunas")
print(f"SoftFailure : {df_soft.shape[0]:>10,} linhas | {df_soft.shape[1]} colunas")
print(f"Lightpath   : {df_lp.shape[0]:>10,} linhas | {df_lp.shape[1]} colunas")

HardFailure :     65,733 linhas | 9 colunas
SoftFailure :     53,697 linhas | 9 colunas
Lightpath   :  2,721,600 linhas | 8 colunas


### Preparação dos dados

#### Junção DFS

In [34]:
# Padroniza rótulo de falha
# Hard/Soft usam coluna "Failure" (binária); renomeia para alinhar com Lightpath
for df in [df_hard, df_soft]:
    df.rename(columns={"Failure": "Failure_type"}, inplace=True)

# Converte BER do Lightpath de dB para razão linear
# Hard/Soft: BER é razão linear (ex: 2.28e-08)
# Lightpath: BER está em dB logarítmico (ex: -267 dB)
# Fórmula: BER_ratio = 10^(BER_dB / 10)
# Valores muito negativos (ex: -267 dB) resultam em ~0, indicando operação sem erros
df_lp["BER"] = 10 ** (df_lp["BER_dB"] / 10)
df_lp.drop(columns=["BER_dB"], inplace=True)

# Concatena os três datasets
# sort=False preserva a ordem das colunas; colunas ausentes viram NaN (tratado depois)
df = pd.concat([df_hard, df_soft, df_lp], ignore_index=True, sort=False)

print(f"Dataset unificado: {df.shape[0]:,} linhas × {df.shape[1]} colunas")
print(f"\nDistribuição por fonte:\n{df['source'].value_counts().to_string()}")

Dataset unificado: 2,841,030 linhas × 12 colunas

Distribuição por fonte:
source
lightpath       2721600
hard_failure      65733
soft_failure      53697


#### Conversão para datetime

In [35]:
# Converte Timestamp para datetime
# Hard/Soft: Unix epoch em segundos → datetime UTC
# Lightpath:  índice sequencial (1, 2, 3...) → datetime relativo ao início do Hard Failure
#             (garante um eixo temporal comparável entre os três datasets)
mask_hs  = df["source"].isin(["hard_failure", "soft_failure"])
lp_start = pd.to_datetime(df.loc[mask_hs, "Timestamp"].astype(int).min(), unit="s")

df.loc[mask_hs,  "Timestamp"] = pd.to_datetime(df.loc[mask_hs,  "Timestamp"].astype(int), unit="s")
df.loc[~mask_hs, "Timestamp"] = lp_start + pd.to_timedelta(df.loc[~mask_hs, "Timestamp"].astype(int), unit="s")
df["Timestamp"] = pd.to_datetime(df["Timestamp"])

# Ordena cronologicamente
# Importante para interpolação linear e para modelos de série temporal posteriores
df.sort_values("Timestamp", inplace=True)
df.reset_index(drop=True, inplace=True)

print(f"Período coberto: {df['Timestamp'].min()}  →  {df['Timestamp'].max()}")

Período coberto: 2021-06-11 06:57:14  →  2021-06-23 22:14:11


#### Replice Failure - NaN = 0


In [36]:
# Preenche rótulo ausente com 0 (Normal)
# NaN em Failure_type indica ausência de evento de falha → operação normal
df["Failure_type"] = df["Failure_type"].fillna(0).astype("int8")  # int8 economiza memória

print("Distribuição de Failure_type (dados originais):")
vc = df["Failure_type"].value_counts().sort_index()
for cls, cnt in vc.items():
    label = {0:"Normal", 1:"Falha/ECL", 2:"EDFA", 3:"NLI"}.get(cls, str(cls))
    print(f"  Classe {cls} [{label:10s}]: {cnt:>10,}  ({cnt/len(df)*100:.1f}%)")

Distribuição de Failure_type (dados originais):
  Classe 0 [Normal    ]:    792,219  (27.9%)
  Classe 1 [Falha/ECL ]:    688,011  (24.2%)
  Classe 2 [EDFA      ]:    680,400  (23.9%)
  Classe 3 [NLI       ]:    680,400  (23.9%)


### Interpolação - Tratamento de NaN nas Features Numéricas

In [37]:
# Interpolação intra-série
# Preenche NaN *dentro de cada fonte* usando interpolação linear.
# Agrupa por "source" para não misturar séries temporais distintas.
# limit_direction="both" trata NaN no início e fim de cada grupo.
# Atenção: NaN *entre fontes* (ex: InputPower em linhas do Lightpath) permanecem
# e serão tratados pelo IterativeImputer na próxima etapa.
numeric_cols = ["BER", "OSNR", "InputPower", "OutputPower",
                "LP_length_km", "Laser_current_mA", "LP_power_dBm"]

df[numeric_cols] = (
    df.groupby("source")[numeric_cols]
      .transform(lambda s: s.interpolate(method="linear", limit_direction="both"))
)

remaining = df[numeric_cols].isna().sum()
remaining = remaining[remaining > 0]
print("NaN estruturais restantes (cross-source — serão tratados pelo IterativeImputer):")
print(remaining.to_string())

NaN estruturais restantes (cross-source — serão tratados pelo IterativeImputer):
InputPower          2721600
OutputPower         2721600
LP_length_km         119430
Laser_current_mA     119430
LP_power_dBm         119430


### Imputação ML — IterativeImputer (MICE + ExtraTreesRegressor)

In [ ]:
# Colunas categóricas não têm representação numérica → "N/A"
df["Type"] = df["Type"].fillna("N/A")
df["ID"]   = df["ID"].fillna("N/A")

# Amostra estratificada por Failure_type para o fit do imputer
# (treina em IMPUTER_SAMPLE linhas, aplica no dataset completo — muito mais rápido)
sample_idx = (
    df.groupby("Failure_type", group_keys=False)
      .apply(lambda g: g.sample(
          n=min(len(g), IMPUTER_SAMPLE // df["Failure_type"].nunique()),
          random_state=RANDOM_STATE
      ))
      .index
)
df_sample = df.loc[sample_idx, IMPUTE_COLS]

imputer = IterativeImputer(
    estimator=ExtraTreesRegressor(n_estimators=10, random_state=RANDOM_STATE, n_jobs=-1),
    max_iter=5,
    random_state=RANDOM_STATE,
    verbose=1
)

print(f"Treinando imputer em {len(df_sample):,} amostras...")
imputer.fit(df_sample)

print(f"Aplicando imputer em {len(df):,} linhas...")
df[IMPUTE_COLS] = imputer.transform(df[IMPUTE_COLS])

print(f"\nNulos restantes nas features numéricas: {df[IMPUTE_COLS].isna().sum().sum()}")

### Aumento de Dados

Estratégia em duas etapas:
1. **SMOTE** — gera amostras sintéticas das classes minoritárias (falhas) para equilibrar a base
2. **Ruído Gaussiano** — adiciona pequenas perturbações às amostras originais para aumentar a diversidade

In [39]:
# Diagnóstico pré-augmentation
# Com o imputer já aplicado, todas as colunas numéricas estão completas.
# Agora todas as features podem ser usadas no SMOTE.
smote_features = IMPUTE_COLS  # usa a constante global definida na configuração

X = df[smote_features].values
y = df["Failure_type"].values

print(f"Features disponíveis para augmentation: {smote_features}")
print(f"\nDistribuição antes do aumento:")
for cls, cnt in zip(*np.unique(y, return_counts=True)):
    label = {0:"Normal", 1:"Falha/ECL", 2:"EDFA", 3:"NLI"}.get(cls, str(cls))
    print(f"  Classe {cls} [{label:10s}]: {cnt:>10,}  ({cnt/len(y)*100:.1f}%)")

Features disponíveis para augmentation: ['BER', 'OSNR', 'InputPower', 'OutputPower', 'LP_length_km', 'Laser_current_mA', 'LP_power_dBm']

Distribuição antes do aumento:
  Classe 0 [Normal    ]:    792,219  (27.9%)
  Classe 1 [Falha/ECL ]:    688,011  (24.2%)
  Classe 2 [EDFA      ]:    680,400  (23.9%)
  Classe 3 [NLI       ]:    680,400  (23.9%)


In [40]:
# SMOTE — balanceamento das classes de falha
# O SMOTE gera amostras sintéticas interpolando k vizinhos mais próximos da
# classe minoritária. Aplicado APÓS a imputação para usar todas as 7 features.
#
# Estratégia: eleva cada classe de falha até a mesma quantidade da classe Normal,
# garantindo distribuição equilibrada sem duplicar amostras reais.

class_counts  = dict(zip(*np.unique(y, return_counts=True)))
majority_count = class_counts[0]  # classe Normal é a referência

# Só reamostra classes que ainda estão abaixo da Normal
sampling_strategy = {
    cls: majority_count
    for cls, cnt in class_counts.items()
    if cls != 0 and cnt < majority_count
}

if sampling_strategy:
    smote = SMOTE(sampling_strategy=sampling_strategy, random_state=RANDOM_STATE, k_neighbors=5)
    X_sm, y_sm = smote.fit_resample(X, y)

    df_smote_new = pd.DataFrame(
        X_sm[len(df):],          # apenas as linhas *novas* geradas pelo SMOTE
        columns=smote_features
    )
    df_smote_new["Failure_type"] = y_sm[len(df):].astype("int8")
    df_smote_new["source"]       = "synthetic_smote"
    print(f"SMOTE gerou {len(df_smote_new):,} amostras sintéticas")
else:
    df_smote_new = pd.DataFrame(columns=df.columns)
    print("Classes já balanceadas — SMOTE não necessário")

SMOTE gerou 327,846 amostras sintéticas


In [41]:
# Ruído Gaussiano — expansão até TARGET_ROWS
# Calcula quantas linhas ainda faltam para atingir a meta e gera exatamente esse número.
# O ruído é proporcional ao desvio padrão de cada coluna (escala = NOISE_SCALE * std),
# preservando a distribuição estatística original enquanto adiciona diversidade.
# O Timestamp é gerado como offset aleatório dentro do intervalo original.

df_after_smote = pd.concat([df, df_smote_new], ignore_index=True)
rows_needed     = max(0, TARGET_ROWS - len(df_after_smote))

print(f"Linhas após SMOTE : {len(df_after_smote):,}")
print(f"Linhas necessárias: {rows_needed:,}  (meta: {TARGET_ROWS:,})")

if rows_needed > 0:
    rng      = np.random.default_rng(RANDOM_STATE)
    col_stds = df[smote_features].std().values

    # Amostra base para o ruído: mantém proporção de classes
    base_sample = df.sample(n=rows_needed, replace=True, random_state=RANDOM_STATE)

    noise = rng.normal(loc=0, scale=NOISE_SCALE, size=(rows_needed, len(smote_features)))
    noise *= col_stds  # escala coluna a coluna

    df_noise = base_sample[smote_features].values + noise
    df_noise = pd.DataFrame(df_noise, columns=smote_features)
    df_noise["Failure_type"] = base_sample["Failure_type"].values
    df_noise["source"]       = "synthetic_gaussian"

    # Timestamp: distribuição uniforme dentro do intervalo original
    t_min = df["Timestamp"].astype("int64").min()
    t_max = df["Timestamp"].astype("int64").max()
    df_noise["Timestamp"] = pd.to_datetime(
        rng.integers(t_min, t_max, size=rows_needed)
    )

    print(f"Ruído gaussiano gerou {len(df_noise):,} amostras")
else:
    df_noise = pd.DataFrame(columns=df.columns)
    print("Meta já atingida — ruído gaussiano não necessário")

Linhas após SMOTE : 3,168,876
Linhas necessárias: 0  (meta: 5,000)
Meta já atingida — ruído gaussiano não necessário


In [42]:
# Consolida: original + SMOTE sintético + gaussiano
df_final = pd.concat([df_after_smote, df_noise], ignore_index=True, sort=False)

# Garante tipos corretos na base final
df_final["Failure_type"] = df_final["Failure_type"].astype("int8")
df_final["Type"]         = df_final["Type"].fillna("N/A")
df_final["ID"]           = df_final["ID"].fillna("N/A")

print(f"{'='*50}")
print(f"Base final: {df_final.shape[0]:,} linhas × {df_final.shape[1]} colunas")
print(f"{'='*50}")
print(f"\nDistribuição de Failure_type:")
for cls, cnt in df_final["Failure_type"].value_counts().sort_index().items():
    label = {0:"Normal", 1:"Falha/ECL", 2:"EDFA", 3:"NLI"}.get(cls, str(cls))
    print(f"  Classe {cls} [{label:10s}]: {cnt:>10,}  ({cnt/len(df_final)*100:.1f}%)")
print(f"\nDistribuição por source:")
print(df_final["source"].value_counts().to_string())

Base final: 3,168,876 linhas × 12 colunas

Distribuição de Failure_type:
  Classe 0 [Normal    ]:    792,219  (25.0%)
  Classe 1 [Falha/ECL ]:    792,219  (25.0%)
  Classe 2 [EDFA      ]:    792,219  (25.0%)
  Classe 3 [NLI       ]:    792,219  (25.0%)

Distribuição por source:
source
lightpath          2721600
synthetic_smote     327846
hard_failure         65733
soft_failure         53697


### Exportação da Base

In [45]:
csv_path = OUTPUT_DIR / "dataset_unificado.csv"
parquet_path = OUTPUT_DIR / "dataset_unificado.parquet"

df_final.to_csv(csv_path, index=False)
csv_mb = csv_path.stat().st_size / 1e6
print(f"CSV    → {csv_path}  ({csv_mb:.1f} MB)")

try:
    df_final.to_parquet(parquet_path, index=False, engine="fastparquet")
except Exception:
    try:
        df_final.to_parquet(parquet_path, index=False, engine="pyarrow")
    except Exception as e:
        print(f"Parquet não salvo ({e.__class__.__name__}): reinicie o kernel e tente novamente")
        parquet_path = None

if parquet_path and parquet_path.exists():
    pq_mb = parquet_path.stat().st_size / 1e6
    print(f"Parquet→ {parquet_path}  ({pq_mb:.1f} MB)")

print(f"\nResumo final:")
print(df_final.describe())

CSV    → ..\Datasets\output\dataset_unificado.csv  (490.8 MB)
Parquet→ ..\Datasets\output\dataset_unificado.parquet  (113.2 MB)

Resumo final:
                           Timestamp           BER          OSNR  \
count                        2841030  3.168876e+06  3.168876e+06   
mean   2021-06-11 12:50:40.884103168  7.793902e-04  1.842714e+01   
min              2021-06-11 06:57:14  0.000000e+00  0.000000e+00   
25%              2021-06-11 07:01:09  4.171583e-10  1.600954e+01   
50%              2021-06-11 07:05:04  4.218674e-07  1.800339e+01   
75%              2021-06-11 07:08:59  4.057551e-05  2.025556e+01   
max              2021-06-23 22:14:11  1.147182e-01  3.860000e+01   
std                              NaN  3.681812e-03  4.206401e+00   

         InputPower   OutputPower  Failure_type  LP_length_km  \
count  3.168876e+06  3.168876e+06  3.168876e+06  3.168876e+06   
mean  -1.981357e+01  6.571314e-01  1.500000e+00  1.927255e+03   
min   -3.800000e+01  4.000000e-01  0.000000e+00  